# Recording Presence And Breathing Triage

Purpose: inspect a local repo recording and ask two narrow questions: does the current heuristic see person-like presence, and is there a breathing-band rhythm worth investigating?

Run path: from the repo root, start Jupyter with `uv run jupyter lab` or run cells in your editor. Set `RECORDING_PATH` below to your sample CSV, or leave it as `None` to auto-pick the first local `data/recordings/**/*_csi.csv` file.

Fixture / simulated source: this notebook only searches inside this repo's `data/recordings/` tree. The current local demo capture is `data/recordings/esp32_csi/dummy-test/rx01_desk_csi.csv`.

Expected interpretation: presence is a windowed CSI disturbance score. Breathing is a weaker spectral diagnostic, so treat it as exploratory unless the capture has one stable link, enough duration, and a known quiet setup.

Limitations: multiple people are not separable from one scalar breathing waveform. Several breathers can appear as several spectral peaks only when their rates and spatial/link signatures differ; otherwise the signal is a mixture and cannot assign breaths to people without multistatic links, array structure, or a trained source-separation model.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import analyze_capture, load_esp32_capture
    from ruview.vitals.breathing import extract_breathing_residual
    from ruview.vitals.preprocessing import CsiVitalPreprocessor
    from ruview.vitals.quality import estimate_rate_from_samples
except Exception as exc:
    raise RuntimeError('Run this notebook from the repo environment, e.g. `uv run jupyter lab`.') from exc

In [ ]:
def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


repo_root = find_repo_root()

# Recording source. Leave RECORDING_CSV as None to auto-pick a local CSV.
USE_RECORDING = True
RECORDING_CSV = None

recording_candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
if not USE_RECORDING:
    raise ValueError('This triage notebook is recording-backed; keep USE_RECORDING=True or use notebooks 01-06 for synthetic fixtures.')

recording_path = Path(RECORDING_CSV).expanduser().resolve() if RECORDING_CSV else None
if recording_path is None and recording_candidates:
    recording_path = recording_candidates[0]
if recording_path is None:
    raise FileNotFoundError('No local *_csi.csv recording found under data/recordings/.')

print(f'Repo root: {repo_root}')
print(f'Selected recording: {recording_path}')
print('Local recording candidates:')
for candidate in recording_candidates[:10]:
    print('  ', candidate.relative_to(repo_root))

In [ ]:
capture = load_esp32_capture(recording_path)
summary, predictions = analyze_capture(capture, window_size=64, step_size=16)

summary_dict = json.loads(summary.to_json())
print(json.dumps(summary_dict, indent=2))

presence_threshold = 0.35
person_like = summary.predicted_state in {'still', 'moving'} or summary.presence_score_max >= presence_threshold
print('\nPerson-like presence detected by current heuristic:', person_like)
print(f"State={summary.predicted_state}, mean_presence={summary.presence_score_mean:.3f}, max_presence={summary.presence_score_max:.3f}")

In [ ]:
def elapsed_seconds(capture):
    real_ts = capture.timestamps
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = capture.host_monotonic_ns.astype(np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


def filled_amplitude(capture, indices):
    amp = capture.amplitude[indices].astype(np.float64, copy=True)
    mask = capture.valid_mask[indices]
    amp[~mask] = np.nan
    valid = np.isfinite(amp)
    counts = valid.sum(axis=0)
    sums = np.where(valid, amp, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    amp[rows, cols] = col_means[cols]
    return amp


def scalar_residual_stream(capture, indices):
    amp = filled_amplitude(capture, indices)
    preprocessor = CsiVitalPreprocessor(n_subcarriers=amp.shape[1], alpha=0.05)
    samples = []
    for frame_amplitude in amp:
        residuals = preprocessor.process(frame_amplitude)
        samples.append(float(extract_breathing_residual(residuals if residuals is not None else [])))
    return np.asarray(samples, dtype=np.float64)


def stream_indices_for_key(capture, key):
    mac, channel = key
    return np.asarray([
        idx for idx, row in enumerate(capture.rows)
        if row.get('mac', '') == mac and row.get('channel', '') == channel
    ], dtype=int)


link_keys = [(row.get('mac', ''), row.get('channel', '')) for row in capture.rows]
link_counts = Counter(link_keys)
print('Top local links in this capture:')
for (mac, channel), count in link_counts.most_common(8):
    print(f'  mac={mac:17s} channel={channel:>2s} packets={count}')

In [ ]:
def resample_for_breathing(times, values):
    order = np.argsort(times)
    times = np.asarray(times, dtype=np.float64)[order]
    values = np.asarray(values, dtype=np.float64)[order]
    keep = np.isfinite(times) & np.isfinite(values)
    times, values = times[keep], values[keep]
    if times.size == 0:
        return times, values, 0.0, 0.0, 0.0
    unique = np.concatenate([[True], np.diff(times) > 1e-9])
    times, values = times[unique], values[unique]
    if times.size < 4:
        return times, values, 0.0, 0.0, 0.0
    duration = float(times[-1] - times[0])
    effective_rate = (times.size - 1) / duration if duration > 0 else 0.0
    dts = np.diff(times)
    median_rate = 1.0 / float(np.median(dts[dts > 0])) if np.any(dts > 0) else 0.0
    if effective_rate <= 0.0:
        return times, values, effective_rate, median_rate, duration
    grid = np.linspace(times[0], times[-1], max(int(round(duration * effective_rate)) + 1, times.size))
    return grid, np.interp(grid, times, values), effective_rate, median_rate, duration


def top_breathing_peaks(samples, sample_rate_hz, band=(0.1, 0.5), top_k=5):
    samples = np.asarray(samples, dtype=np.float64)
    if samples.size < 4 or sample_rate_hz <= 0.0:
        return []
    low, high = band
    high = min(high, 0.45 * sample_rate_hz)
    if high <= low:
        return []
    centered = samples - float(np.mean(samples))
    n_fft = 1 << (max(centered.size, 4) - 1).bit_length()
    padded = np.zeros(n_fft, dtype=np.float64)
    padded[: centered.size] = centered * np.hanning(centered.size)
    spectrum = np.fft.rfft(padded)
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / sample_rate_hz)
    power = spectrum.real * spectrum.real + spectrum.imag * spectrum.imag
    mask = (freqs >= low) & (freqs <= high)
    candidates = np.flatnonzero(mask)
    peaks = []
    for idx in candidates:
        if idx == 0 or idx >= power.size - 1:
            continue
        if power[idx] >= power[idx - 1] and power[idx] >= power[idx + 1]:
            peaks.append((float(freqs[idx] * 60.0), float(power[idx])))
    peaks.sort(key=lambda item: item[1], reverse=True)
    return peaks[:top_k]


def estimate_breathing_for_indices(name, indices):
    times = elapsed_seconds(capture)[indices]
    scalar = scalar_residual_stream(capture, indices)
    grid, signal, effective_rate, median_rate, duration = resample_for_breathing(times, scalar)
    high = min(0.5, 0.45 * effective_rate)
    if signal.size < 4 or duration < 10.0 or high <= 0.1:
        estimate = None
    else:
        estimate = estimate_rate_from_samples(
            signal,
            sample_rate_hz=effective_rate,
            band_hz=(0.1, high),
            min_duration_seconds=10.0,
            min_samples=max(int(round(10.0 * effective_rate)), 4),
        )
    return {
        'name': name,
        'packet_count': int(len(indices)),
        'duration_seconds': duration,
        'effective_rate_hz': effective_rate,
        'median_gap_rate_hz': median_rate,
        'times': grid,
        'signal': signal,
        'estimate': estimate,
        'candidate_peaks_bpm': top_breathing_peaks(signal, effective_rate),
    }


dominant_key = link_counts.most_common(1)[0][0]
streams = [
    estimate_breathing_for_indices('dominant_link', stream_indices_for_key(capture, dominant_key)),
    estimate_breathing_for_indices('all_packets_mixed_links', np.arange(len(capture.rows))),
]

for result in streams:
    est = result['estimate']
    print('\n', result['name'])
    print(f"  packets={result['packet_count']} duration={result['duration_seconds']:.2f}s effective_rate={result['effective_rate_hz']:.2f}Hz median_gap_rate={result['median_gap_rate_hz']:.2f}Hz")
    if est is None or not est.is_available:
        print('  breathing estimate: unavailable')
    else:
        print(f"  breathing estimate: {est.value_bpm:.1f} BPM, confidence={est.confidence:.2f}, status={est.status.value}")
    print('  top breathing-band peaks (BPM, relative power):')
    for bpm, power in result['candidate_peaks_bpm']:
        print(f'    {bpm:5.1f} BPM  power={power:.3g}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)

mid = [(p.start_seconds + p.end_seconds) / 2.0 for p in predictions]
axes[0].plot(mid, [p.presence_score for p in predictions], label='presence score')
axes[0].plot(mid, [p.motion_score for p in predictions], label='motion score')
axes[0].axhline(0.35, color='black', linestyle='--', linewidth=1, label='presence threshold')
axes[0].set_title('Windowed person/presence heuristic')
axes[0].set_xlabel('seconds')
axes[0].set_ylabel('score')
axes[0].legend()

for result in streams:
    axes[1].plot(result['times'], result['signal'], label=result['name'], alpha=0.85)
axes[1].set_title('Scalar breathing residual streams')
axes[1].set_xlabel('seconds')
axes[1].set_ylabel('residual amplitude')
axes[1].legend()

for result in streams:
    signal = result['signal']
    rate = result['effective_rate_hz']
    if signal.size < 4 or rate <= 0.0:
        continue
    centered = signal - float(np.mean(signal))
    n_fft = 1 << (max(centered.size, 4) - 1).bit_length()
    padded = np.zeros(n_fft, dtype=np.float64)
    padded[: centered.size] = centered * np.hanning(centered.size)
    spectrum = np.fft.rfft(padded)
    freqs_bpm = np.fft.rfftfreq(n_fft, d=1.0 / rate) * 60.0
    power = spectrum.real * spectrum.real + spectrum.imag * spectrum.imag
    mask = (freqs_bpm >= 6.0) & (freqs_bpm <= 30.0)
    axes[2].plot(freqs_bpm[mask], power[mask], label=result['name'])
axes[2].set_title('Breathing-band spectrum')
axes[2].set_xlabel('breaths per minute')
axes[2].set_ylabel('power')
axes[2].legend();

## Reading Multiple Breathers

If two people breathe at different rates, the mixed CSI residual can show two spectral peaks. That is only a clue. It does not prove two people unless the peaks are stable over time and appear with different spatial/link/subcarrier signatures.

If two people breathe at nearly the same rate, or one person dominates the link, the signal often collapses into one peak. If they move, posture-shift, or packets mix across MAC/channel links, false peaks can appear. The practical path is to estimate presence/count separately, then use per-link or per-subcarrier breathing spectra as supporting evidence.